# 1. Context

This notebook analyzes OCR performance of Google Document AI Solution over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import plotly.express as px

import plotly.graph_objects as go
fig = go.Figure()
from plotly.subplots import make_subplots

In [2]:
results_root = Path("../results/google")

In [3]:
writing_system_to_language = {'Devanagari': ['hindi','sanskrit','nepali','konkani'],
                              'tamil': ['tamil'],'telugu': ['telugu'],'Kannada': ['kannada'],'Malayalam': ['malayalam'],
                              'Bengali': ['bengali', 'assamese', 'manipuri'],'Meetei-mayek': ['manipuri'],'Gujarati': ['gujarati'],
                              'Gurmukhi': ['punjabi'],'Odia': ['oriya'],'Arabic': ['kashmiri', 'sindhi', 'urdu'],
                              'Latin': ['english'],'Ol-chiki': ['santali']}

In [4]:
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [6]:
language_to_writing_system = {
    "marathi": ["Devanagari"], "hindi": ["Devanagari"], "sanskrit": ["Devanagari"],
    "tamil": ["tamil"], "telugu": ["telugu"], "kannada": ["Kannada"],"malayalam": ["Malayalam"],
    "bengali": ["Bengali"], "assamese": ["Bengali"],"manipuri": ["Bengali", "Meetei-mayek"],"nepali": ["Devanagari"],
    "gujarati": ["Gujarati"], "punjabi": ["Gurmukhi"], "konkani": ["Devanagari"],"oriya": ["Odia"],"kashmiri": ["Devanagari", "Arabic"], 
    "sindhi": ["Arabic", "Devanagari"], "urdu": ["Arabic"],"english": ["Latin"], "santali": ["Ol-chiki", "Devanagari"],
    }

In [7]:
writing_sys_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)

In [9]:
script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [10]:
script_language_result

,languages
script,
Gujarati,[gujarati]
Gurmukhi,[punjabi]
Devanagari,"[hindi, konkani, sanskrit, nepali]"
telugu,[telugu]
Bengali,"[assamese, bengali]"
Latin,[english]
Kannada,[kannada]
tamil,[tamil]
Arabic,[sindhi]


In [11]:
def get_language_results_path(script: str, script_language_result: pd.DataFrame) -> list[Path]:
    """Get list of results path for given script"""

    results_script_lang = script_language_result.loc[script].to_list()[0]
    results_script_lang_path = [results_root.joinpath(lang).joinpath('results.csv') for lang in results_script_lang]

    return results_script_lang_path

In [12]:
script = 'Devanagari'
results_devanagari_path = get_language_results_path(script, script_language_result)

# 3. Devanagari 

In [13]:
results_list = []
for result_path in results_devanagari_path:
    lang = result_path.parent.name
    df_res = pd.read_csv(result_path)
    df_res['language'] = lang
    results_list.append(df_res)

results_devanagari = pd.concat(results_list)

In [14]:
lang_list = list(results_devanagari['language'].unique())

fig = make_subplots(rows=results_devanagari['language'].nunique(), 
                    cols=1, 
                    shared_yaxes=True,
                    subplot_titles=lang_list)

degrad_level_cer = ['cer_l0', 'cer_l1', 'cer_l2','cer_l3']

for idx, language in enumerate(lang_list):
    results_lang = results_devanagari.loc[results_devanagari['language'] == language]
    for cer_level in degrad_level_cer:
        fig.add_trace(go.Box(y=results_lang[cer_level] * 100,
                                name=cer_level,
                                text=results_lang['file_id']), 
                                row=idx + 1, col=1)


fig.update_layout(height=1200, width=1400)
fig.update_layout(showlegend=False, title_text=f"CER % across various degradation level in {script} languages")
fig.update_yaxes(title_text="CER in %")
fig.show()


In [15]:
lang_list = list(results_devanagari['language'].unique())

fig = make_subplots(rows=results_devanagari['language'].nunique(), 
                    cols=1, 
                    shared_yaxes=True,
                    subplot_titles=lang_list)

degrad_level_wer = ['wer_l0', 'wer_l1', 'wer_l2','wer_l3']

for idx, language in enumerate(lang_list):
    results_lang = results_devanagari.loc[results_devanagari['language'] == language]
    for wer_level in degrad_level_wer:
        fig.add_trace(go.Box(y=results_lang[wer_level] * 100,
                                name=wer_level,
                                text=results_lang['file_id']), 
                                row=idx + 1, col=1)


fig.update_layout(height=1200, width=1400)
fig.update_layout(showlegend=False, title_text=f"WER % across various degradation level and {script} languages")
fig.update_yaxes(title_text="WER in %")
fig.show()